# Processes, Jobs & Signals

## What a process actually is

A **process** is what you get when a program is running. The program itself — a file on disk — is just a recipe. The moment the kernel loads it into memory and starts executing it, you have a process.

Every process has a small bundle of attributes the kernel tracks:

- **PID (Process ID)** — a unique number identifying this process while it's alive. Reused after the process exits, but never two live processes share a PID.
- **PPID (Parent PID)** — the PID of the process that started this one. Every process has a parent (except PID 1, the init process — see below).
- **UID (User ID)** — the user account this process runs as. Determines what files it can read/write.
- **GID (Group ID)** — the group the process belongs to.
- **Command** — the program being run, plus its arguments.
- **State** — running, sleeping, stopped, or zombie (covered below).

A useful analogy: a process is like a **conversation**. The program is the script. The process is what happens when someone reads the script aloud — it has an identity (PID), a parent who started it talking (PPID), and a state (still talking, paused, done).

**PID 1 is special.** It's the first program the kernel starts at boot. On modern Linux, that's almost always `systemd` (full coverage in notebook 09). Every other process is a descendant of PID 1, directly or indirectly. If a process's parent dies while the child is still running, the kernel **re-parents** the child to PID 1, which "adopts" it.

Two more things to know:

- **Every command you run is a new process.** When you type `ls /etc`, the shell creates a new `ls` process, waits for it to finish, and shows the prompt again. Pipelines (`a | b | c`) create one process per stage, all running in parallel.
- **The shell itself is a process.** When you log in, a new shell process starts. When you `exit` or close the terminal, that process ends.

## `ps` — viewing processes

**`ps`** ("process status") gives you a **snapshot** of what's running right now. It doesn't update — for that, use `top` or `htop` (next sections).

`ps` has two flag styles that coexist for historical reasons. Both work; both are widely used:

**BSD style** — flags without a dash. The classic combination is **`ps aux`**:

```
ps aux
```

- `a` — show processes of all users
- `u` — show user-friendly output (with CPU, memory, etc.)
- `x` — include processes not attached to a terminal (daemons)

**SysV style** — flags with a dash. The classic combination is **`ps -ef`**:

```
ps -ef
```

- `-e` — show every process
- `-f` — full format (with extra columns)

Both produce nearly identical information. Pick one and stick with it.

The columns you'll see most:

| Column | Meaning |
|---|---|
| **`PID`** | the process ID |
| **`PPID`** | the parent's PID |
| **`USER`** or **`UID`** | who owns this process |
| **`%CPU`** | CPU usage averaged over lifetime (not instantaneous) |
| **`%MEM`** | percentage of RAM in use |
| **`RSS`** | resident set size — KB of RAM actually in use |
| **`VSZ`** | virtual size — KB of address space mapped (always bigger than RSS) |
| **`STAT`** | state letter (R/S/D/T/Z, see next section) |
| **`TTY`** | controlling terminal, or `?` for no terminal |
| **`TIME`** | accumulated CPU time |
| **`COMMAND`** or **`CMD`** | the command being run |

For custom columns: **`ps -eo pid,ppid,user,%cpu,%mem,comm`**. Specify exactly what you want, in any order. `man ps` lists all available column names.

**Process tree.** Add **`--forest`** to draw the parent-child tree inline:

```bash
ps -ef --forest | head -30
```

Or use the dedicated **`pstree`** command:

```bash
pstree -p              # show PIDs
```

In [ ]:
!ps aux | head -10
!ps -ef | head -10
!ps -eo pid,user,%cpu,%mem,comm --sort=-%mem | head -10
!pstree -p 2>/dev/null | head -10

## `top` — the live view

`top` is the classic real-time process viewer. Type **`top`**, and you get a screen that refreshes every few seconds showing CPU, memory, and the busiest processes.

The top of the screen has summary info — load average, task counts, CPU usage breakdown, memory usage. Below that is a table of processes, sorted by CPU usage by default.

The keystrokes that make `top` actually useful (press them inside `top`):

| Key | Effect |
|---|---|
| **`q`** | quit |
| **`h`** or **`?`** | help screen |
| **`P`** | sort by CPU usage |
| **`M`** | sort by memory (RSS) |
| **`T`** | sort by accumulated CPU time |
| **`1`** | toggle per-CPU lines (vs aggregate) |
| **`c`** | toggle full command line |
| **`u`** | filter by user (prompts for name) |
| **`k`** | send a signal to a process (interactive `kill`) |
| **`r`** | renice a process (change priority) |
| **`W`** | save your current config to `~/.toprc` |

For headless monitoring (capturing output for logs), use **batch mode**:

```bash
top -b -n 1                # one iteration, no interactive screen
top -b -n 1 | head -20     # just the summary + top 10 processes
```

This is the form you use in scripts and cron jobs.

## `htop` — the friendlier alternative

**`htop`** is `top`'s much friendlier cousin. Coloured, mouse-aware, with a clearer layout. It's not installed by default on most distros — install it:

```bash
sudo apt install htop          # Debian/Ubuntu
sudo dnf install htop          # RHEL/Fedora
```

What `htop` adds:

- **Per-CPU bars** at the top, with memory and swap gauges.
- **Function-key shortcuts** at the bottom — no need to memorise letters:
  - **F5** — tree view (toggle)
  - **F6** — pick which column to sort by
  - **F9** — send a signal to the highlighted process (with a menu!)
  - **F10** or `q` — quit
- **Scroll up/down** through the full process list with arrow keys.
- **Spacebar** — tag a process. Tagged processes get bulk actions.
- **Mouse support** — click columns to sort, click processes to select.

For an LFCS-level workflow: install `htop` on every machine you regularly use. It's strictly better than `top` for interactive use. `top` is still useful for scripted monitoring (batch mode is more reliable).

For a more modern alternative still, look at **`btop`** — even prettier, with graphs over time.

## Process states — R, S, D, T, Z

In `ps` output the **`STAT`** column shows a single letter for the process's current state. Five states matter:

| Letter | Name | What it means |
|---|---|---|
| **`R`** | Running | currently on a CPU, or runnable and waiting for one |
| **`S`** | Sleeping | waiting for an event (most idle processes) — responds to signals |
| **`D`** | Uninterruptible sleep | waiting on the kernel — usually disk I/O. **Can't be killed.** |
| **`T`** | Stopped | paused by a signal (Ctrl-Z) or by a debugger |
| **`Z`** | Zombie | finished, but the parent hasn't collected its exit status yet |

In practice:

- **Almost every process is `S` most of the time.** Sleeping isn't bad — it just means the process is waiting (for input, for a timer, for a network packet, etc.). A CPU with 200 processes mostly idle is normal.
- **`R` doesn't mean "currently running on a CPU"** — it means runnable. If your CPU is busy, many processes can be in state `R` waiting their turn.
- **`D` is the dangerous one.** A process stuck in `D` is waiting on the kernel and **can't be killed** — not even with `SIGKILL`. Almost always caused by a hung disk, a wedged NFS mount, or a faulty driver. If you see persistent `D`, investigate the I/O system. Reboot fixes it but doesn't tell you the root cause.
- **`Z` (zombie) is harmless individually**, dangerous in quantity. A zombie holds only a PID and an exit code. A well-behaved parent process collects (`wait()`s for) its children promptly. If you see hundreds of zombies under one parent, that parent has a bug.

Trailing letters you'll see on the state — extra modifiers:

| Modifier | Meaning |
|---|---|
| **`+`** | in the foreground process group (i.e. currently controlling the terminal) |
| **`s`** | session leader |
| **`l`** | multi-threaded |
| **`<`** | high priority (negative nice) |
| **`N`** | low priority (positive nice) |

So `Ss+` is a sleeping session leader in the foreground — what your interactive shell looks like.

In [ ]:
!ps -eo state,comm | sort | uniq -c | sort -rn

## Foreground vs background — `&`, `jobs`, `fg`, `bg`

By default, when you run a command, the shell waits for it to finish before showing the next prompt. That command is running in the **foreground**.

To run a command in the **background** — let it run while you keep typing — append **`&`**:

```bash
sleep 60 &
[1] 12345                  # the shell prints [job number] PID
```

Now you have your prompt back while `sleep 60` runs in the background. To see what's running in the background of your current shell:

```bash
jobs
```

You'll see a list with **job numbers** (the `[1]`, `[2]`, ...) which are local to your shell. To bring a background job back to the foreground:

```bash
fg %1                      # bring job 1 to foreground
fg                         # bring the most recent job to foreground (no need to specify)
```

To pause the foreground process and put it in the background:

1. Press **Ctrl-Z** while it's running. The shell stops the process (state `T`) and shows you the prompt.
2. Type **`bg`** to resume the stopped process in the background.

This is the classic "I started something that's taking too long" recovery:

```
$ long-running-command
^Z                          # press Ctrl-Z
[1]+  Stopped               long-running-command
$ bg                        # send to background
[1]+ long-running-command &
$                           # prompt is back, command keeps running
```

**Ctrl-C** is different — it **terminates** the foreground process by sending `SIGINT`. **Ctrl-Z** just pauses it (sends `SIGTSTP`).

To wait for all background jobs of your shell to finish:

```bash
wait
```

To wait for a specific one:

```bash
wait %1
wait 12345                 # by PID
```

In [ ]:
!bash -c 'sleep 5 & sleep 3 & jobs; wait'

## Surviving session loss — `nohup`, `disown`, `setsid`, and `tmux`

When you log out (or your SSH session drops), the shell sends `SIGHUP` ("hangup") to all its jobs by default. Most programs treat `SIGHUP` as "you should exit" — which means closing the terminal also kills your background jobs.

For long-running tasks, you need one of these:

**`nohup`** — runs a command with `SIGHUP` ignored, and redirects stdin from `/dev/null` and stdout/stderr to `nohup.out`:

```bash
nohup long-running-script.sh &
```

The command keeps running even if the shell dies. Output is appended to `nohup.out` in the current directory (or `~/nohup.out` if the current directory isn't writable).

**`disown`** — removes a job from the shell's job table. The shell forgets about it and won't `SIGHUP` it on exit. Useful for promoting an already-running background job:

```bash
some-command &
disown %1                  # forget about job 1
# or just:
disown                     # forget about the most recent job
```

**`setsid`** — runs a command in a new session, fully detached from the controlling terminal. The cleanest "daemonise" primitive:

```bash
setsid long-running-script.sh &
```

The command becomes its own session leader, with no controlling terminal — even closing the shell won't affect it.

**The modern answer: terminal multiplexers — `tmux` or `screen`.**

Instead of detaching individual commands, run a **multiplexer**: a long-running session that hosts shells inside it. Your SSH session dies; the multiplexer keeps running; you SSH back in and **attach** to your session and pick up exactly where you left off.

```bash
tmux                        # start a new session
# do work...
# press Ctrl-B then D to detach
tmux attach                 # come back to the session
tmux ls                     # list available sessions
```

Same idea with `screen`:

```bash
screen
screen -r                   # reattach
```

For most LFCS-flavoured work, **`tmux` is the right answer**. `nohup` and `disown` are still useful for quick one-offs, but multiplexers handle the whole problem better.

## Signals — the kernel's notification system

A **signal** is a small message the kernel delivers to a process. Each signal has a number and a name (e.g. signal 15 is `SIGTERM`). Signals are how you ask a process to do something asynchronously — terminate, reload config, pause, stop.

When a signal arrives, the receiving process can either:

1. **Run its default handler** (usually "terminate") — what happens if the process didn't set up anything special.
2. **Run a custom handler** — code the program author wrote to handle the signal (e.g. SIGHUP → reload config).
3. **Ignore the signal** — pretend it didn't happen.
4. **Block the signal** — queue it for later.

Two signals **cannot be caught, blocked, or ignored**. The kernel enforces them no matter what:

- **`SIGKILL` (signal 9)** — terminate immediately. No cleanup. The "last resort" kill.
- **`SIGSTOP` (signal 19)** — pause execution immediately. Resume with `SIGCONT`.

Every other signal can be handled.

To see all the signals on your system:

```bash
kill -l
```

You'll see ~30 named signals. Most you'll never use; the handful that matter day-to-day are below.

## The signals you'll actually use

A handful covers 95% of real-world use:

| Signal | Number | Default | Use when |
|---|---|---|---|
| **`SIGTERM`** | 15 | terminate | **The polite shutdown.** Default for `kill`. The process is asked to exit cleanly. Catchable — apps can clean up first. |
| **`SIGKILL`** | 9 | terminate | **The last resort.** The process is killed immediately with no chance to clean up. Use when `SIGTERM` is ignored. |
| **`SIGINT`** | 2 | terminate | What **Ctrl-C** sends. Same intent as SIGTERM but tied to keyboard interaction. |
| **`SIGHUP`** | 1 | terminate | Originally "hangup" (modem dropped). Now **conventionally used to tell daemons to reload their config** — `systemctl reload` on many services sends SIGHUP. |
| **`SIGQUIT`** | 3 | core dump | What **Ctrl-\** sends. Terminate with a core dump (for debugging). |
| **`SIGSTOP`** | 19 | stop | Pause the process. Cannot be caught. |
| **`SIGTSTP`** | 20 | stop | What **Ctrl-Z** sends. Pause, but **catchable** — apps can override. |
| **`SIGCONT`** | 18 | resume | Wake a stopped process. |
| **`SIGUSR1`** | 10 | terminate | Application-defined. Many daemons use it to mean "rotate logs" or "reload." |
| **`SIGUSR2`** | 12 | terminate | Another application-defined slot. |

You'll send the top three constantly:
- **`SIGTERM`** for "please shut down" (the default for `kill`)
- **`SIGKILL`** when `SIGTERM` doesn't work
- **`SIGHUP`** to tell a daemon to reread its config

## `kill`, `killall`, `pkill` — sending signals

Three commands send signals. They differ in **how you specify the target**.

**`kill PID`** — sends a signal to a process by PID. Default signal is `SIGTERM`. Specify a different signal with `-SIGNAL` (number or name):

```bash
kill 12345                  # SIGTERM to PID 12345
kill -TERM 12345            # same thing, explicit
kill -15 12345              # same again, by number
kill -KILL 12345            # SIGKILL — the forceful version
kill -9 12345               # SIGKILL by number — what people usually type
kill -HUP 12345             # tell the process to reload config
```

`kill -l` lists all signal names. `kill -l SIGTERM` shows the number for a name; `kill -l 15` shows the name for a number.

**`killall NAME`** — sends a signal to all processes matching a name. Useful when you know the program but not the PID:

```bash
killall nginx               # SIGTERM to every nginx process
killall -9 chrome           # SIGKILL to every chrome process
killall -HUP rsyslogd       # reload all rsyslog instances
```

`killall` matches the exact basename of the executable. To match a substring or pattern, use `pkill` (below).

**`pkill PATTERN`** — like `killall` but the pattern can be a regex on the command line:

```bash
pkill -f long-script.sh     # match against the full command line (-f)
pkill -u alice              # all processes owned by alice
pkill -SIGTERM bash         # all bash processes
```

The companion `pgrep` returns matching PIDs **without** killing them — useful for inspection first:

```bash
pgrep -af bash              # list all bash processes (name + full command)
pgrep -u alice              # PIDs of alice's processes
```

**Workflow tip:** before killing, **use `pgrep` first to verify** you're hitting the right targets. Once you're satisfied with the list, swap `pgrep` for `pkill` (same flags).

In [ ]:
!bash -c 'sleep 30 & pid=$!; sleep 0.2; kill -TERM $pid; wait $pid 2>/dev/null; echo "exit code: $?"'
!pgrep -af bash | head -5

## The polite-then-forceful escalation

The standard pattern for shutting down a misbehaving process is **escalation**:

1. Send **`SIGTERM`** first.
2. **Wait a few seconds** to give the process a chance to clean up — flush logs, close database connections, save state.
3. If it's still alive, send **`SIGKILL`**.

```bash
kill -TERM 12345
sleep 5
if kill -0 12345 2>/dev/null; then    # check if still alive
    kill -KILL 12345
fi
```

(The `kill -0` is a trick — it sends signal 0, which is a "test only" signal that checks whether the process exists without actually signalling it.)

**Don't reach for `kill -9` first.** A common new-user mistake. `SIGKILL` doesn't let the process clean up — files may be left half-written, databases corrupted, locks held, temp files orphaned. Always try `SIGTERM` first, give it a few seconds, then escalate if needed.

This is exactly what **`systemctl stop`** does for you under the hood (notebook 09): sends `SIGTERM` to the service, waits for `TimeoutStopSec` (default 90s), then sends `SIGKILL` if the process is still alive. You rarely have to do the escalation manually — but you should know what's happening.

## Priority — `nice` and `renice`

Linux's scheduler distributes CPU time among runnable processes. By default everything gets a fair share. You can tilt the scales with **nice values**.

**Nice value range: `-20` (highest priority) to `19` (lowest priority).** Default is `0`. Counterintuitive — *higher* nice means *lower* priority. The mental model: nice means "willing to yield." A nice=19 process is **very willing** to step aside; a nice=-20 process is selfish about CPU.

**Only root can decrease nice** (raise priority). Any user can increase nice (lower priority on their own processes). This prevents users from giving themselves more CPU than they're due.

To **start a command with a specific nice**:

```bash
nice -n 10 some-command        # start at nice=10 (lower priority)
nice -n -5 some-command        # nice=-5 — needs sudo
```

To **change the priority of a running process** (renice):

```bash
renice 5 -p 12345              # set PID 12345 to nice=5
renice -n -5 -p 12345          # lower nice (root only)
renice 10 -u alice             # nice all of alice's processes to 10
```

There's also **`ionice`** for I/O priority (separate from CPU priority). Three I/O classes:

```bash
ionice -c 3 some-command       # class 3 = idle (only do I/O when nothing else needs it)
ionice -c 2 -n 7 some-command  # class 2 (best-effort), niceness 7 (within class)
```

Useful when running a heavy backup or rsync that you don't want to choke production:

```bash
nice -n 19 ionice -c 3 rsync -av source/ dest/
```

For LFCS-level work: know `nice` and `renice`. `ionice` is a nice-to-have.

In [ ]:
!nice -n 10 bash -c 'echo "my nice level is: $(awk "{print \$19}" /proc/self/stat)"'

## Scheduled tasks — `cron`

The classic scheduler. **`cron`** runs commands at specified times. Each user has their own **crontab** (cron table) — a file listing scheduled jobs.

To edit your crontab:

```bash
crontab -e                  # edit (opens $EDITOR)
crontab -l                  # list
crontab -r                  # remove (careful — no confirmation)
```

A crontab line has **six fields**:

```
* * * * * command-to-run
│ │ │ │ │
│ │ │ │ └── day of week (0-7, Sunday is 0 or 7)
│ │ │ └──── month (1-12)
│ │ └────── day of month (1-31)
│ └──────── hour (0-23)
└────────── minute (0-59)
```

Each field can be:

- A specific number — `30`
- A range — `9-17`
- A list — `1,15,30`
- A step — `*/15` (every 15 units)
- A wildcard `*` (any value)

Examples:

```cron
# Every minute
* * * * * /usr/local/bin/check.sh

# Every 15 minutes
*/15 * * * * /usr/local/bin/check.sh

# Daily at 2:30 AM
30 2 * * * /usr/local/bin/backup.sh

# Weekdays at 9 AM
0 9 * * 1-5 /usr/local/bin/morning-report.sh

# First of every month
0 0 1 * * /usr/local/bin/monthly-cleanup.sh
```

A few critical gotchas:

- **`cron` runs with a minimal environment.** `$PATH` is often just `/usr/bin:/bin`. If your script uses commands in `/usr/local/bin`, give the full path or set `PATH` at the top of your crontab.
- **No interactive shell.** Anything that prompts will hang silently. Scripts must be non-interactive.
- **Output is emailed** (if configured) or lost. Redirect explicitly: `>> /var/log/myjob.log 2>&1`.
- **Test outside cron first.** Run the script manually as the user that will own the cron job — if it works there but fails in cron, it's almost always an environment difference.

There are also **system-wide crontabs** in `/etc/crontab` and `/etc/cron.d/`, which have an extra **user** field after the time fields (e.g. `30 2 * * * root /usr/local/bin/backup.sh`). User crontabs (`crontab -e`) don't need the user column.

## Other schedulers — `at` and systemd timers

**`at`** — schedule a **one-off** command at a future time:

```bash
echo "/usr/local/bin/cleanup.sh" | at now + 1 hour
echo "/usr/local/bin/reboot.sh"  | at 03:00 tomorrow
at 2pm Friday <<< "/usr/local/bin/report.sh"

atq                                  # list pending at jobs
atrm 5                               # remove at job 5
```

`at` accepts very flexible time specifications: `now + 30 minutes`, `midnight`, `noon`, `tomorrow`, `next Tuesday`, `2025-12-25 09:00`. Many distros don't install `at` by default — `sudo apt install at` or `sudo dnf install at`.

**systemd timers** — the modern replacement for `cron` on systemd-based systems (i.e. almost everywhere). Timers are described as `.timer` unit files in `/etc/systemd/system/`, paired with a `.service` unit that does the actual work.

```ini
# /etc/systemd/system/backup.timer
[Unit]
Description=Daily backup

[Timer]
OnCalendar=daily              # human-readable time spec
Persistent=true               # run after boot if a scheduled run was missed

[Install]
WantedBy=timers.target
```

```ini
# /etc/systemd/system/backup.service
[Unit]
Description=Backup job

[Service]
Type=oneshot
ExecStart=/usr/local/bin/backup.sh
```

Then:

```bash
sudo systemctl enable --now backup.timer
systemctl list-timers          # see all timers + when they next fire
```

**Advantages of systemd timers over cron:**
- Logs to `journalctl` (one place for all output)
- `Persistent=true` catches up missed runs (e.g. if the machine was off)
- Rich scheduling syntax (`OnBootSec`, `OnUnitActiveSec`, monotonic timers)
- Resource controls (limit CPU/memory of scheduled jobs)
- First-class testing — `systemctl start backup.service` runs it immediately

For LFCS, **know `cron` deeply and `systemd timers` at preview level**. Full coverage of timers and the rest of systemd comes in notebook 09.